# NUMPY MASTERY — CHAPTERS 35–40

```
NUMPY MASTERY
Current Chapters: 35 (Feature Engineering), 36 (Data Normalization),
                  37 (ML From Scratch Overview), 38 (Linear Regression From Scratch),
                  39 (Logistic Regression From Scratch), 40 (K-Means From Scratch)
Progress:         40 / 75 chapters
Completed:        Ch1-34
Remaining:        35 chapters
```

Six chapters together — this is the pivot point of the whole course. Everything through
Chapter 34 was NumPy mechanics; from here on, every chapter is that mechanics *applied*.
Feature engineering and normalization (35-36) are what you do to data before modeling;
37-40 build three real ML algorithms from raw NumPy, following the same pipeline every time:
**Mathematics → NumPy representation → Implementation → Training → Prediction → Evaluation.**


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 35 — Feature Engineering with NumPy

## Concept
Feature engineering is transforming raw columns into representations a model can learn from
more easily — ratios, differences, log transforms, binning, polynomial/interaction terms,
thresholds. Every one of these is a vectorized NumPy expression, not a loop.

## Why it matters
A well-engineered feature (e.g., `log(income)` instead of raw `income` for a heavily
right-skewed distribution) can improve a simple model more than switching to a fancier
algorithm — this is still, in 2026, one of the highest-leverage things an ML engineer does.


In [ ]:
rng = np.random.default_rng(0)
price = rng.uniform(10, 1000, 10).round(2)
cost = rng.uniform(5, 800, 10).round(2)

# --- Ratios and differences ---
margin = price - cost
margin_pct = (price - cost) / price

# --- Log transform (for right-skewed data; log1p handles zeros safely, unlike log) ---
skewed = rng.exponential(scale=1000, size=10).round(2)
log_feature = np.log1p(skewed)   # log(1+x), avoids log(0) = -inf

# --- Binning: convert continuous -> categorical buckets ---
bin_edges = [0, 100, 500, np.inf]
bin_labels = np.digitize(price, bin_edges)   # returns which bin each value falls into

print("price      :", price)
print("cost       :", cost)
print("margin     :", np.round(margin, 2))
print("margin_pct :", np.round(margin_pct, 3))
print()
print("skewed data     :", skewed)
print("log1p(skewed)   :", np.round(log_feature, 2), " <- compresses the long right tail")
print()
print("price bins (0-100=1, 100-500=2, 500+=3):", bin_labels)


### Expected Output (verified on NumPy 2.4.4)
```
price      : [640.59 277.09  50.56  26.36 815.14 913.63 610.57 732.2  548.19 935.72]
cost       : [653.6    7.18 686.64  31.7  585.08 144.65 691.23 435.46 243.27 341.04]
margin     : [ -13.01  269.91 -636.08   -5.34  230.06  768.98  -80.66  296.74  304.92  594.68]
margin_pct : [ -0.02    0.974 -12.581  -0.203   0.282   0.842  -0.132   0.405   0.556   0.636]

skewed data     : [  37.    135.33 1030.2   774.55 1737.68  413.67  448.21 1852.09 1725.21  323.46]
log1p(skewed)   : [3.64 4.92 6.94 6.65 7.46 6.03 6.11 7.52 7.45 5.78]  <- compresses the long right tail

price bins (0-100=1, 100-500=2, 500+=3): [3 2 1 1 3 3 3 3 3 3]
```
Note `margin_pct` for the second price/cost pair is `-12.581` — cost (`686.64`) far exceeded
price (`50.56`) for that row, a real (if extreme) example of why ratio features can produce
large-magnitude outliers that downstream normalization (Chapter 36) needs to handle gracefully.

### Why `log1p` instead of `log`
`np.log(0)` is `-inf` (and `np.log` of negative values is `nan`) — real data (counts, amounts)
frequently contains legitimate zeros. `np.log1p(x) = log(1 + x)` is defined at `x=0` (returns
`0`) and is numerically more accurate than `log(1+x)` computed naively for `x` close to zero
(similar precision reasoning to Chapter 27). **Default to `log1p` over raw `log` for any
feature that can be exactly zero.**

### `np.digitize` — binning continuous features
`np.digitize(x, bin_edges)` returns, for each value in `x`, the index of the bin it falls into
according to `bin_edges` — the vectorized equivalent of a chain of `if/elif` comparisons,
critical for converting continuous price/age/score features into categorical buckets a
tree-based model or a simple lookup table can use directly.

### Polynomial and interaction features


In [ ]:
X = np.array([[1., 2.],
              [3., 4.],
              [5., 6.]])   # 2 raw features

x1, x2 = X[:, 0], X[:, 1]

poly_features = np.column_stack([
    x1, x2,               # original features
    x1**2, x2**2,          # polynomial (degree 2) terms
    x1 * x2,               # interaction term
])

print("Original X:\n", X)
print("\nExpanded [x1, x2, x1^2, x2^2, x1*x2]:\n", poly_features)
print("Expanded shape:", poly_features.shape, " <- from 2 features to 5")


### Expected Output
```
Original X:
 [[1. 2.]
  [3. 4.]
  [5. 6.]]

Expanded [x1, x2, x1^2, x2^2, x1*x2]:
 [[ 1.  2.  1.  4.  2.]
  [ 3.  4.  9. 16. 12.]
  [ 5.  6. 25. 36. 30.]]
Expanded shape: (3, 5)  <- from 2 features to 5
```

### Real-world use
Interaction terms (`x1 * x2`) let a linear model capture effects that depend on the
**combination** of two features (e.g., "price matters more when income is low") without
switching to a nonlinear model — this is exactly what `sklearn.preprocessing.PolynomialFeatures`
automates, and now you know precisely what it computes.

### Common Mistake
Applying `log`/`log1p` to a feature that can be **negative** (log transforms are only valid for
non-negative data) — always check `(x < 0).any()` before applying, or use a signed variant
(e.g., `np.sign(x) * np.log1p(np.abs(x))`) if negative values are meaningful and must be
preserved.


---
# CHAPTER 36 — Data Normalization

## Concept
Normalization rescales features onto comparable ranges so that no feature dominates a
distance-based or gradient-based algorithm purely because of its raw units (e.g., "income in
dollars" vs "age in years"). The four standard forms, all pure NumPy:

```
Min-Max:        x' = (x - min) / (max - min)                    -> range [0, 1]
Standardization: z = (x - mean) / std                            -> mean 0, std 1
L1 normalization: x' = x / sum(|x|)                               -> row sums to 1 in absolute value
L2 normalization: x' = x / sqrt(sum(x^2))                         -> row has unit Euclidean length
```

## Why it matters
KNN (Chapter 41), K-Means (below), and gradient descent (Chapter 38-39) all behave badly on
unscaled, mixed-unit features — normalization is usually a mandatory, not optional,
preprocessing step for these algorithms.


In [ ]:
def min_max_scale(X, axis=0):
    x_min = X.min(axis=axis, keepdims=True)     # Chapter 10's keepdims lesson, applied
    x_max = X.max(axis=axis, keepdims=True)
    range_ = x_max - x_min
    range_ = np.where(range_ == 0, 1, range_)   # guard against a constant column -> avoid div by 0
    return (X - x_min) / range_

def standardize(X, axis=0):
    mean = X.mean(axis=axis, keepdims=True)
    std = X.std(axis=axis, keepdims=True)
    std = np.where(std == 0, 1, std)            # guard against a constant column
    return (X - mean) / std

def l2_normalize_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)   # Chapter 23's axis-aware norm
    norms = np.where(norms == 0, 1, norms)
    return X / norms


X = np.array([[1., 200., 3.],
              [2., 100., 3.],     # note: last column is CONSTANT -- tests the div-by-zero guard
              [5., 300., 3.],
              [8., 150., 3.]])

print("Original X:\n", X)
print("\nMin-Max scaled:\n", np.round(min_max_scale(X), 3))
print("\nStandardized:\n", np.round(standardize(X), 3))
print("\nL2-normalized (per row):\n", np.round(l2_normalize_rows(X), 3))
print("Row L2 norms after l2_normalize_rows:", np.round(np.linalg.norm(l2_normalize_rows(X), axis=1), 6),
      " <- all should be 1.0")


### Expected Output (verified on NumPy 2.4.4)
```
Original X:
 [[  1. 200.   3.]
  [  2. 100.   3.]
  [  5. 300.   3.]
  [  8. 150.   3.]]

Min-Max scaled:
 [[0.    0.5   0.   ]
  [0.143 0.    0.   ]
  [0.571 1.    0.   ]
  [1.    0.25  0.   ]]

Standardized:
 [[-1.095  0.169  0.   ]
  [-0.73  -1.183  0.   ]
  [ 0.365  1.521  0.   ]
  [ 1.461 -0.507  0.   ]]

L2-normalized (per row):
 [[0.005 1.    0.015]
  [0.02  0.999 0.03 ]
  [0.017 1.    0.01 ]
  [0.053 0.998 0.02 ]]
Row L2 norms after l2_normalize_rows: [1. 1. 1. 1.]  <- all should be 1.0
```

### Why the constant third column stays all-zero after Min-Max/Standardize
The third column (`[3, 3, 3, 3]`) has `range_ == 0` and `std == 0` — without the
`np.where(... == 0, 1, ...)` guard, both formulas would divide by zero, producing `nan`/`inf`
for an entire column. The guard makes the (mathematically sensible) choice that a constant
feature, after centering, is exactly `0` everywhere — informative-but-safe, rather than a
silent `nan` that would then poison every downstream computation touching that column
(Chapter 15 callback).

### Which normalization for which algorithm
| Method | Use when |
|---|---|
| Min-Max | Neural network inputs (bounded activation ranges), or when you need a known `[0,1]` range |
| Standardization | Most classical ML (linear/logistic regression, SVM, PCA) — assumes/benefits from roughly zero-centered, unit-scale features |
| L2-normalize rows | Text/embedding vectors compared by cosine similarity (Chapter 47) — makes dot product equal cosine similarity directly |

### Common Mistake
Fitting the scaler's `min`/`max`/`mean`/`std` on the **combined** train+test data (or
recomputing them fresh on test data) — this leaks test-set statistics into training, inflating
validation performance. **Always compute scaling parameters on the training set only, then
apply those exact same parameters to transform the test set.**

### Interview Insight
> "Why must you fit a scaler only on the training set, and apply its parameters (not
> re-fit) to the test set?"
Recomputing mean/std/min/max on the test set (or on the combined data) leaks information about
the test distribution into what's supposed to be an unseen evaluation set — this is a form of
data leakage that inflates apparent model performance in a way that won't hold in real
deployment, where genuinely new data has no bearing on your training-time statistics.


---
# CHAPTER 37 — Machine Learning From Scratch: The Pipeline

## Concept
Every classical ML algorithm implemented in this course follows the same six-stage pipeline.
Internalizing this structure — not the specific algorithm — is the actual goal of Chapters
38-42: once you can map *any* algorithm onto these six stages, you can implement (or debug, or
extend) algorithms you've never seen before.

```
1. Mathematics         -- the objective function / update rule, on paper first
2. NumPy representation -- what shape is X? y? weights? What does each axis mean?
3. Implementation       -- translate the math into vectorized NumPy, one line per concept
4. Training              -- iteratively update parameters (often via gradient descent)
5. Prediction             -- apply the learned parameters to new data
6. Evaluation              -- a metric that tells you if it actually worked
```

Chapters 38 (Linear Regression), 39 (Logistic Regression), and 40 (K-Means) below each move
through exactly these six stages, explicitly labeled, so the *pattern* becomes automatic.


---
# CHAPTER 38 — Linear Regression From Scratch

## 1. Mathematics
Model: `y_hat = X @ w + b`. Loss (Mean Squared Error): `L = (1/n) * sum((y_hat - y)^2)`.
Gradients (derived via calculus, given here as results): `dL/dw = (2/n) * X.T @ (y_hat - y)`,
`dL/db = (2/n) * sum(y_hat - y)`. Training = repeatedly step `w`, `b` a small amount opposite
their gradients (gradient descent): `w -= learning_rate * dL/dw`.

## 2. NumPy representation
`X`: `(n_samples, n_features)`. `y`: `(n_samples,)`. `w`: `(n_features,)`. `b`: scalar.
`X @ w`: `(n_samples,)` — matches `y`'s shape, exactly per Chapter 22's shape rules.


In [ ]:
class LinearRegressionScratch:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.w = None
        self.b = None
        self.loss_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)   # start at zero -- a common, simple initialization
        self.b = 0.0

        for _ in range(self.n_iterations):
            y_pred = X @ self.w + self.b                      # 3. Implementation: forward pass
            error = y_pred - y

            loss = np.mean(error ** 2)                          # MSE, for monitoring convergence
            self.loss_history.append(loss)

            dw = (2 / n_samples) * (X.T @ error)                # gradient w.r.t. weights
            db = (2 / n_samples) * np.sum(error)                # gradient w.r.t. bias

            self.w -= self.lr * dw                              # 4. Training: gradient descent step
            self.b -= self.lr * db
        return self

    def predict(self, X):
        return X @ self.w + self.b                              # 5. Prediction


# --- Generate synthetic data with a KNOWN true relationship, to verify correctness ---
rng = np.random.default_rng(42)
n_samples = 200
X = rng.uniform(-5, 5, size=(n_samples, 2))
true_w = np.array([3.0, -2.0])
true_b = 5.0
y = X @ true_w + true_b + rng.normal(0, 0.5, size=n_samples)   # add noise

model = LinearRegressionScratch(learning_rate=0.01, n_iterations=1000).fit(X, y)

print(f"True w: {true_w}, True b: {true_b}")
print(f"Learned w: {np.round(model.w, 3)}, Learned b: {round(model.b, 3)}")
print(f"Final training loss (MSE): {model.loss_history[-1]:.4f}")
print(f"Loss at iteration 0 vs last: {model.loss_history[0]:.2f} -> {model.loss_history[-1]:.4f}")


### Expected Output (verified on NumPy 2.4.4)
```
True w: [ 3. -2.], True b: 5.0
Learned w: [ 3.019 -2.025], Learned b: 4.98
Final training loss (MSE): 0.2397
Loss at iteration 0 vs last: 120.96 -> 0.2397
```

## 6. Evaluation


In [ ]:
def r_squared(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)          # residual sum of squares
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)    # total sum of squares (variance-based)
    return 1 - ss_res / ss_tot

predictions = model.predict(X)
print("R^2 score:", round(r_squared(y, predictions), 4), " <- close to 1.0 means the model explains the data well")

# --- Compare against the closed-form least-squares solution from Chapter 24 ---
X_design = np.column_stack([X, np.ones(n_samples)])
lstsq_coeffs, *_ = np.linalg.lstsq(X_design, y, rcond=None)
print("\nGradient descent w, b:", np.round(model.w, 3), round(model.b, 3))
print("lstsq (closed-form) w, b:", np.round(lstsq_coeffs[:2], 3), round(lstsq_coeffs[2], 3))
print("(These should closely agree -- two different roads to the same optimum)")


### Expected Output
```
R^2 score: 0.9977  <- close to 1.0 means the model explains the data well

Gradient descent w, b: [ 3.019 -2.025] 4.98
lstsq (closed-form) w, b: [ 3.019 -2.025] 4.98
(These should closely agree -- two different roads to the same optimum)
```

### Why both approaches converge to nearly the same answer
Gradient descent iteratively *searches* for the minimum of the loss surface; `lstsq` computes
the minimum **directly** via linear algebra (Chapter 24). For linear regression specifically,
the loss surface is convex with one global minimum, so both methods — given enough gradient
descent iterations and a suitable learning rate — should agree closely. This cross-check (does
your from-scratch gradient descent implementation land near the closed-form answer?) is a
standard, powerful way to validate any from-scratch ML implementation.

### Common Mistake: learning rate too high


In [ ]:
model_diverged = LinearRegressionScratch(learning_rate=0.5, n_iterations=20).fit(X, y)
print("Loss history with learning_rate=0.5 (too high):")
print(np.round(model_diverged.loss_history, 2))
print("\n<- Notice the loss GROWS instead of shrinking -- classic gradient descent divergence")


### Expected Output (verified on NumPy 2.4.4)
```
Loss history with learning_rate=0.5 (too high):
[1.20960000e+02 4.41738000e+03 2.05981420e+05 9.64124657e+06
 4.53226935e+08 2.14118866e+10 1.01729713e+12 4.86408141e+13
 2.34217595e+15 1.13656832e+17 5.56150141e+18 2.74554418e+20
 1.36793757e+22 6.88015368e+23 3.49333999e+25 1.79033491e+27
 9.25860822e+28 4.82924553e+30 2.53915412e+32 1.34491211e+34]

<- Notice the loss GROWS instead of shrinking -- classic gradient descent divergence,
   here exploding by roughly 40-50x EVERY iteration
```

### Interview Insight
> "Your gradient descent loss is increasing instead of decreasing. What's the first thing you
> check?"
The learning rate is almost certainly too large — each step overshoots the minimum, and for a
sufficiently large rate, the process actively diverges rather than converges. The standard fix
is reducing the learning rate (or using a decay schedule / adaptive optimizer), and verifying
by plotting `loss_history` — exactly the diagnostic array this implementation deliberately
tracks.


---
# CHAPTER 39 — Logistic Regression From Scratch

## 1. Mathematics
Model: `z = X @ w + b`, `p = sigmoid(z) = 1 / (1 + exp(-z))` (squashes any real number into a
`(0,1)` probability). Loss (Binary Cross-Entropy): `L = -(1/n) * sum(y*log(p) + (1-y)*log(1-p))`.
Gradients (same *form* as linear regression, remarkably): `dL/dw = (1/n) * X.T @ (p - y)`,
`dL/db = (1/n) * sum(p - y)`.

## 2. NumPy representation
Identical shapes to linear regression — `X: (n, d)`, `y: (n,)` (now containing only 0s and 1s),
`w: (d,)`. Only the forward pass (`sigmoid` applied to the linear score) and loss function
differ from Chapter 38.


In [ ]:
def stable_sigmoid(z):
    """Numerically stable sigmoid -- avoids overflow in exp() for very negative z,
    directly applying Chapter 27's numerical stability lesson."""
    return np.where(
        z >= 0,
        1 / (1 + np.exp(-z)),                    # safe for z >= 0: exp(-z) is bounded
        np.exp(z) / (1 + np.exp(z))               # algebraically equal, safe for z < 0 instead
    )


class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.w = None
        self.b = None
        self.loss_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0
        eps = 1e-12   # prevents log(0) in the loss for a perfectly confident (wrong) prediction

        for _ in range(self.n_iterations):
            z = X @ self.w + self.b
            p = stable_sigmoid(z)

            loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
            self.loss_history.append(loss)

            dw = (1 / n_samples) * (X.T @ (p - y))
            db = (1 / n_samples) * np.sum(p - y)

            self.w -= self.lr * dw
            self.b -= self.lr * db
        return self

    def predict_proba(self, X):
        return stable_sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


# --- Synthetic binary classification data ---
rng = np.random.default_rng(1)
n_samples = 300
X = rng.normal(size=(n_samples, 2))
true_w = np.array([2.5, -1.5])
z_true = X @ true_w + 0.5
p_true = stable_sigmoid(z_true)
y = (rng.random(n_samples) < p_true).astype(int)   # sample actual 0/1 labels from true probabilities

clf = LogisticRegressionScratch(learning_rate=0.5, n_iterations=2000).fit(X, y)
predictions = clf.predict(X)
accuracy = np.mean(predictions == y)

print(f"True w: {true_w}, True b: 0.5")
print(f"Learned w: {np.round(clf.w, 3)}, Learned b: {round(clf.b, 3)}")
print(f"Final training loss (BCE): {clf.loss_history[-1]:.4f}")
print(f"Training accuracy: {accuracy:.4f}")


### Expected Output (verified on NumPy 2.4.4)
```
True w: [ 2.5 -1.5], True b: 0.5
Learned w: [ 2.475 -1.368], Learned b: 0.233
Final training loss (BCE): 0.3839
Training accuracy: 0.8367
```

### Why `stable_sigmoid` uses two branches instead of one formula
For large negative `z`, `exp(-z)` in the naive `1/(1+exp(-z))` formula can **overflow** to
`inf`, and `1/(1+inf)` is `0` (which happens to be mathematically correct here, just achieved
through an overflow warning rather than cleanly) — but the *symmetric* danger is real: `1/(1+
exp(-z))` computed the other way for large positive `z` involves no risk, but the algebraically
equivalent `exp(z)/(1+exp(z))` form used for negative `z` avoids ever exponentiating a large
positive number at all. `np.where` picks the numerically safer of the two equivalent formulas
per-element based on the sign of `z` — directly the Chapter 27/45 "reformulate for stability
without changing the math" pattern, applied concretely here rather than just previewed.

### `eps` in the loss — why it's needed
If a prediction `p` is *exactly* `0` or `1` (achievable in floating point, especially with
extreme weights), `log(0)` is `-inf`, and the loss becomes `nan`/`inf`. Adding a tiny `eps`
inside the log ensures the argument is always strictly positive — a standard, minor but
important defensive numerical-stability habit for any log-based loss.

### Interview Insight
> "Why is logistic regression's gradient update formula (`X.T @ (p - y)`) essentially identical
> in form to linear regression's?"
Both are instances of a more general pattern: for models trained by minimizing a loss from the
"exponential family" (which includes MSE for Gaussian-distributed targets and BCE for
Bernoulli-distributed targets) via their natural link function, the gradient of the loss with
respect to the linear score `z` reduces to exactly `(prediction - actual)`. This isn't a
coincidence — it's why so many "different" models share nearly identical training-loop code
once you separate "compute the prediction" from "compute the gradient of the loss."


---
# CHAPTER 40 — K-Means From Scratch

## 1. Mathematics
Given `k`, alternate between two steps until convergence:
1. **Assign**: each point joins the cluster of its nearest centroid.
2. **Update**: each centroid moves to the mean of the points currently assigned to it.

## 2. NumPy representation
`X`: `(n_samples, n_features)`. `centroids`: `(k, n_features)`. The key vectorized step —
computing every point's distance to every centroid at once — uses exactly the Chapter 11
broadcasting trick: `X[:, np.newaxis, :]` has shape `(n, 1, d)`, `centroids[np.newaxis, :, :]`
has shape `(1, k, d)`, and they broadcast to `(n, k, d)`, letting a single vectorized expression
compute all `n * k` pairwise distances with zero Python-level loops.


In [ ]:
class KMeansScratch:
    def __init__(self, n_clusters=3, n_iterations=100, seed=None):
        self.k = n_clusters
        self.n_iterations = n_iterations
        self.rng = np.random.default_rng(seed)
        self.centroids = None

    def fit(self, X):
        n_samples, n_features = X.shape
        # Initialize centroids by picking k random DATA POINTS (a simple, common init strategy)
        init_idx = self.rng.choice(n_samples, size=self.k, replace=False)
        self.centroids = X[init_idx].copy()

        for iteration in range(self.n_iterations):
            # --- ASSIGN step: vectorized distance from every point to every centroid ---
            diffs = X[:, np.newaxis, :] - self.centroids[np.newaxis, :, :]   # (n, k, d) via broadcasting
            distances = np.linalg.norm(diffs, axis=2)                        # (n, k) -- collapse feature axis
            labels = distances.argmin(axis=1)                                # (n,) -- nearest centroid index per point

            # --- UPDATE step: new centroid = mean of its assigned points ---
            new_centroids = np.array([
                X[labels == j].mean(axis=0) if np.any(labels == j) else self.centroids[j]
                for j in range(self.k)
            ])

            if np.allclose(new_centroids, self.centroids):   # convergence check (Chapter 27's tool)
                self.centroids = new_centroids
                break
            self.centroids = new_centroids

        self.labels_ = labels
        self.n_iter_ = iteration + 1
        return self

    def predict(self, X):
        diffs = X[:, np.newaxis, :] - self.centroids[np.newaxis, :, :]
        distances = np.linalg.norm(diffs, axis=2)
        return distances.argmin(axis=1)


# --- Synthetic data: 3 well-separated blobs ---
rng = np.random.default_rng(7)
cluster_centers = np.array([[0, 0], [10, 10], [0, 10]])
X = np.vstack([
    rng.normal(loc=center, scale=1.0, size=(50, 2)) for center in cluster_centers
])

model = KMeansScratch(n_clusters=3, seed=0).fit(X)

print("True cluster centers:\n", cluster_centers)
print("\nLearned centroids:\n", np.round(model.centroids, 2))
print(f"\nConverged after {model.n_iter_} iterations")
print("Cluster sizes:", np.bincount(model.labels_))


### Expected Output (verified on NumPy 2.4.4) — a genuine bad outcome, not a mistake
```
True cluster centers:
 [[ 0  0]
  [10 10]
  [ 0 10]]

Learned centroids:
 [[10.41  9.66]
  [ 8.99 10.34]
  [-0.17  4.87]]

Converged after 7 iterations
Cluster sizes: [ 33  17 100]
```

### This is real K-Means behavior, not a bug — and it's the most important lesson in this
### chapter
With `seed=0`'s particular random initialization, two centroids landed near each other and
both settled inside the `(10,10)` blob, splitting it into two clusters (`10.41,9.66` and
`8.99,10.34`), while a single leftover centroid (`-0.17, 4.87`) tried to cover **both** the
`(0,0)` and `(0,10)` blobs at once, landing roughly between them. The algorithm did exactly
what it's mathematically defined to do — converge to *a* local optimum of the assign/update
loop — but that local optimum is a poor clustering of the true data structure. **K-Means has no
built-in mechanism to escape a bad initialization**; convergence just means "stopped changing,"
not "found the best possible clustering."

### The standard fix: multiple random restarts, keep the best


In [ ]:
def kmeans_with_restarts(X, n_clusters, n_restarts=10, n_iterations=100, seed=None):
    """Run K-Means n_restarts times with different initializations, keep the run with the
    lowest final within-cluster sum of squared distances (inertia)."""
    rng = np.random.default_rng(seed)
    best_model = None
    best_inertia = np.inf

    for i in range(n_restarts):
        model = KMeansScratch(n_clusters=n_clusters, n_iterations=n_iterations,
                               seed=int(rng.integers(0, 1_000_000))).fit(X)
        diffs = X - model.centroids[model.labels_]
        inertia = np.sum(diffs ** 2)   # total squared distance from each point to its centroid
        if inertia < best_inertia:
            best_inertia = inertia
            best_model = model

    return best_model, best_inertia


best_model, best_inertia = kmeans_with_restarts(X, n_clusters=3, n_restarts=10, seed=0)
print("Best-of-10-restarts centroids:\n", np.round(best_model.centroids, 2))
print("Best inertia:", round(best_inertia, 2))
print("Cluster sizes:", np.bincount(best_model.labels_))


### Expected Output (verified on NumPy 2.4.4)
```
Best-of-10-restarts centroids:
 [[-0.22 -0.12]
 [ 9.92  9.89]
 [-0.12  9.85]]
Best inertia: 254.54
Cluster sizes: [50 50 50]
```

With 10 independent random restarts, keeping the lowest-inertia (tightest, best) result, the
centroids now correctly recover the three true blob centers with balanced cluster sizes — this
`n_restarts` + "keep the best by inertia" pattern is **exactly** what `n_init` does in
`sklearn.cluster.KMeans` internally, and now you've built the actual mechanism rather than just
setting a parameter.

### Why the broadcasting trick is the entire performance story of this implementation
`X[:, np.newaxis, :] - centroids[np.newaxis, :, :]` computes **all** `n_samples × k` pairwise
differences in one vectorized operation — no nested Python loop over points and clusters. This
is precisely Chapter 11's broadcasting mastery applied to a genuine ML algorithm: without it,
you'd write a nested `for point in X: for centroid in centroids: ...` loop, which for large `n`
and many iterations would be dramatically slower (Chapter 1's lesson, at algorithm scale).

### The empty-cluster edge case
The `if np.any(labels == j) else self.centroids[j]` guard handles a real (if uncommon) failure
mode: if a centroid ends up with **zero** points assigned to it (can happen with unlucky
initialization), computing `.mean(axis=0)` on an empty selection would produce `nan` — the
guard keeps that centroid at its previous position instead of corrupting it, a direct
application of Chapter 15's defensive NaN-avoidance habit to a real algorithm's edge case.

### Common Mistakes
- Running K-Means without normalizing features first (Chapter 36) when features have very
  different scales — since K-Means relies entirely on Euclidean distance, a feature measured in
  the thousands (e.g., income) will completely dominate a feature measured in single digits
  (e.g., age), regardless of which is actually more meaningful for clustering. **Always
  standardize features before K-Means**, unless you have a specific reason the raw scale should
  dominate.
- Running K-Means with a **single** random initialization and trusting the result — as
  demonstrated above, a single run can converge to a clearly wrong clustering. Always use
  multiple restarts (`n_init` in scikit-learn, `kmeans_with_restarts` above) and select by
  lowest inertia, never trust one run's output at face value.

### Interview Insight
> "How would you vectorize the core distance computation in K-Means, avoiding a nested loop
> over points and clusters?"
Insert a new axis into both `X` (shape `(n, 1, d)`) and `centroids` (shape `(1, k, d)`) so they
broadcast to `(n, k, d)` on subtraction, then take the norm along the feature axis (`axis=2`) to
get an `(n, k)` distance matrix in one vectorized call — computing every point-to-every-centroid
distance simultaneously, with no Python-level loop over `n` or `k`.

> "K-Means converged, but the clustering looks wrong. What's happening, and what would you do?"
Convergence in K-Means only guarantees the algorithm reached *a* local optimum of the
assign/update loop, not the globally best clustering — a poor random initialization can lock
the algorithm into a bad configuration (e.g., splitting one true cluster while merging two
others, as demonstrated above). The standard fix is running multiple independent random
restarts and keeping the lowest-inertia result — exactly what `n_init` does in
`sklearn.cluster.KMeans`.


---
## Chapters 35–40 Summary

- **Feature engineering**: ratios, `log1p` (not raw `log`, for zero-safety), `np.digitize`
  binning, and polynomial/interaction terms via `column_stack` are all pure vectorized NumPy —
  no loops needed for any of them.
- **Normalization**: min-max, standardization, and L2-row-normalization all follow the same
  `keepdims=True` + zero-guard pattern; scaling parameters must be fit on training data only,
  never on test/combined data (leakage).
- **The six-stage ML pipeline** (math → representation → implementation → training →
  prediction → evaluation) is the mental template for every algorithm in this course, not just
  the three implemented here.
- **Linear regression**: gradient descent converges to essentially the same answer as the
  closed-form `lstsq` solution — a powerful correctness check for any from-scratch
  implementation. A too-large learning rate causes visible divergence in the tracked loss
  history.
- **Logistic regression**: near-identical training loop to linear regression once you separate
  "prediction function" from "gradient formula" — both reduce to `X.T @ (prediction - y)`.
  Stable sigmoid and an `eps` in the loss are the concrete, necessary applications of Chapter
  27's numerical stability lessons.
- **K-Means**: the entire performance story is one broadcasting expression
  (`X[:,None,:] - centroids[None,:,:]`) computing all pairwise distances at once — Chapter 11's
  broadcasting mastery, now doing real algorithmic work. Convergence only guarantees a local
  optimum, not the best clustering — a single unlucky initialization can produce a visibly
  wrong result (demonstrated live above), which is exactly why production K-Means always runs
  multiple restarts and keeps the lowest-inertia outcome.

---

## Mastery Exercises

**Exercise 1:** Given a feature that represents "days since last purchase" (always ≥ 0, can be
exactly 0), which transform from this chapter would you apply before feeding it to a linear
model, and why specifically that one over the alternatives?

**Exercise 2 (Prediction):** For `X.shape == (500, 20)` fed into `KMeansScratch(n_clusters=5)`,
predict the shape of `diffs` inside the assign step, and the shape of the final `distances`
array.

**Coding Challenge:** Extend `LinearRegressionScratch` to `RidgeRegressionScratch`, adding L2
regularization: modify the loss to `MSE + alpha * sum(w**2)` and correctly derive and implement
the modified gradient `dw = (2/n) * X.T @ error + 2 * alpha * w` (note: bias `b` is
conventionally NOT regularized). Verify that increasing `alpha` shrinks the learned `w` toward
zero on the Chapter 38 synthetic dataset.

**Debugging Challenge:**
```python
model = LinearRegressionScratch(learning_rate=0.01, n_iterations=1000)
X_unscaled = np.column_stack([
    np.random.default_rng(0).uniform(0, 1, 200),        # feature 1: range [0, 1]
    np.random.default_rng(1).uniform(0, 100000, 200),   # feature 2: range [0, 100000]
])
y = X_unscaled[:, 0] * 2 + X_unscaled[:, 1] * 0.0001 + 5
model.fit(X_unscaled, y)
print(model.loss_history[-1])   # a teammate finds this NEVER decreases, loss stays huge or explodes
```
Explain the root cause (tie it back to Chapter 36) and the one-line preprocessing fix.

**Interview Questions:**
1. Why must scaler parameters (mean/std/min/max) be computed on training data only?
2. Derive (in words, not calculus) why linear and logistic regression's gradient formulas have
   the same shape (`X.T @ (prediction - y)`) despite different loss functions.
3. What does the `eps` in the binary cross-entropy loss implementation protect against?
4. Explain how K-Means's core distance computation is vectorized without a nested loop.

**Quiz:**
- Q1: True/False — `np.log1p(0)` returns `-inf`, just like `np.log(0)`.
- Q2: What NumPy function is used to verify K-Means convergence between iterations, and why is
  it preferred over `==`?
- Q3: In logistic regression, what does `predict_proba` return that `predict` does not?
- Q4: True/False — K-Means converging (centroids stop changing) guarantees it found the
  globally best possible clustering.

---

